In [2]:
import glob
import pandas as pd
from moviepy.editor import VideoFileClip, concatenate_videoclips
import moviepy
from moviepy.editor import *
import pygame
import cv2
import os

# 경로 설정
PATH_r = '/home/kbj/dev_ws/unmanned_store/data/절도_mp4/'
CSV_PATH = '/home/kbj/dev_ws/unmanned_store/data/영상전처리담당.csv'

# 결과 저장 폴더 생성
os.makedirs('result', exist_ok=True)

# 전역 변수 초기화
ab = []  # 이상행동 구간 저장
norm = []  # 정상행동 구간 저장

In [4]:
def make_clip_video(path, save_path, start_t, end_t):
    """
    영상 클립을 추출하여 저장하는 함수
    Args:
        path: 원본 영상 경로
        save_path: 저장할 클립 경로
        start_t: 시작 시간(초)
        end_t: 끝 시간(초)
    """
    clip_video = VideoFileClip(path).subclip(start_t, end_t)
    clip_video.write_videofile(save_path)
    clip_video.close()

def name_file(df, name):
    """담당자별 데이터 필터링"""
    filtered_df = df[df['담당'] == name]
    if filtered_df.empty:
        print(f"경고: 담당자 '{name}'에 해당하는 데이터가 없습니다.")
        print("사용 가능한 담당자:")
        available_people = df['담당'].dropna().unique()
        for person in available_people:
            count = len(df[df['담당'] == person])
            print(f"  - {person}: {count}개 영상")
    return filtered_df

def where_to_where(df, name):
    """제목에서 범위 추출 (현재 미사용)"""
    minn = int(df['제목'][0].split('~')[0])
    maxx = int(df['제목'][0].split('~')[1])
    return minn, maxx

def abinput(vdo):
    st = int(input('이상행동 시작구간을 입력하세요 (초): '))
    fin = st + 10
    global ab
    for v in vdo:
        filename = v.split('/')[-1]
        ab.append([filename, st, fin])
    return ab[-len(vdo):]

def norminput(vdo):
    st = int(input('정상행동 시작구간을 입력하세요 (초): '))
    fin = st + 10
    global norm
    for v in vdo:
        filename = v.split('/')[-1]
        norm.append([filename, st, fin])
    return norm[-len(vdo):]


    

In [18]:
# 담당자 이름 입력
per = input('담당 인원의 이름을 입력하세요: ')

# CSV 파일 로드 및 필터링
df = pd.read_csv(CSV_PATH)
print("전체 데이터 개수:", len(df))
print("\n담당자별 영상 개수:")
담당자_통계 = df['담당'].value_counts(dropna=False)
print(담당자_통계)

df = name_file(df, per)

if df.empty:
    print("처리할 데이터가 없습니다. 프로그램을 종료합니다.")
    exit()

df = df.drop('제목', axis=1, errors='ignore')  # 컬럼이 없을 경우 에러 방지
df.reset_index(drop=True, inplace=True)

print(f"\n담당자 {per}의 영상 목록 (총 {len(df)}개):")
print(df.head(10))  # 처음 10개만 표시

전체 데이터 개수: 191

담당자별 영상 개수:
담당
NaN      156
김성욱       11
강민지       10
강동욱        4
김범진6       1
김범진7       1
김범진8       1
김범진9       1
김범진10      1
김범진2       1
김범진1       1
김범진3       1
김범진4       1
김범진5       1
Name: count, dtype: int64

담당자 김범진10의 영상 목록 (총 1개):
                     name  n     담당
0  C_3_12_11_BU_DYA_07-27  3  김범진10


In [19]:
# 처리할 영상 개수 설정
num_videos_to_process = int(input(f"총 {len(df)}개 영상 중 처리할 개수를 입력하세요 (전체: {len(df)}): ") or len(df))
num_videos_to_process = min(num_videos_to_process, len(df))

for i in range(num_videos_to_process):
    ab = []  # ✅ 이 줄 추가
    norm = []  # ✅ 이 줄 추가
    print(f"\n=== {i+1}번째 영상 처리 중: {df['name'][i]} ===")
    
    # 해당 이름의 영상 파일들 찾기
    search_pattern = PATH_r + df['name'][i] + '_*.mp4'
    vdo = glob.glob(search_pattern)
    
    if not vdo:
        print(f"경고: {df['name'][i]}에 해당하는 영상 파일을 찾을 수 없습니다.")
        print(f"검색 패턴: {search_pattern}")
        # 해당 폴더의 파일 목록 일부 표시
        all_files = glob.glob(PATH_r + '*.mp4')[:5]
        print(f"예시 파일들: {all_files}")
        continue
    
    print(f"찾은 영상 파일들 ({len(vdo)}개):")
    for v in vdo:
        print(f"  - {v.split('/')[-1]}")
    
    # Pygame 초기화 및 화면 설정
    pygame.init()
    screen_width = 512
    screen_height = 349 
    screen = pygame.display.set_mode((screen_width, screen_height))
    pygame.display.set_caption(df['name'][i])
    
    # FPS 설정
    clock = pygame.time.Clock()
    dt = clock.tick(3)  # 게임화면의 초당 프레임 수를 설정
    print("FPS: " + str(clock.get_fps()))
    
    # 폰트 설정
    game_font = pygame.font.Font(None, 40)
    
    # 시간 관련 설정
    start_ticks = pygame.time.get_ticks()
    elapsed_time = (pygame.time.get_ticks() - start_ticks) / 1000
    total_time = 100
    timer = game_font.render(str(int(elapsed_time)), True, (255, 255, 255))
    screen.blit(timer, (100, 100))
    
    # 첫 번째 영상의 첫 60초 미리보기
    print("영상 미리보기를 시작합니다... (60초)")
    clip_video = VideoFileClip(vdo[0]).subclip(0, 60)
    clip_video.preview()
    clip_video.close()
    
    pygame.quit()
    
    # (Pygame 종료 이후 코드)

    # 이상행동 구간 입력
    N_ab = int(input('입력할 이상행동 구간 개수를 알려주세요: '))
    for j in range(N_ab):
        print(f"\n--- {j+1}번째 이상행동 구간 입력 ---")
        abinput(vdo)

    # 정상행동 구간 입력
    N_norm = int(input('입력할 정상행동 구간 개수를 알려주세요: '))
    for j in range(N_norm):
        print(f"\n--- {j+1}번째 정상행동 구간 입력 ---")
        norminput(vdo)

print(f"\n입력 완료! 총 {len(ab)}개의 이상행동 구간과 {len(norm)}개의 정상행동 구간이 입력되었습니다.")
print(f"현재까지 ab 구간 수: {len(ab)}")
print(f"현재까지 norm 구간 수: {len(norm)}")



=== 1번째 영상 처리 중: C_3_12_11_BU_DYA_07-27 ===
찾은 영상 파일들 (3개):
  - C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2.mp4
  - C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2.mp4
  - C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2.mp4
FPS: 0.0
영상 미리보기를 시작합니다... (60초)
Interrupt

--- 1번째 이상행동 구간 입력 ---

--- 2번째 이상행동 구간 입력 ---

--- 3번째 이상행동 구간 입력 ---

--- 1번째 정상행동 구간 입력 ---

--- 2번째 정상행동 구간 입력 ---

--- 3번째 정상행동 구간 입력 ---

입력 완료! 총 9개의 이상행동 구간과 9개의 정상행동 구간이 입력되었습니다.
현재까지 ab 구간 수: 9
현재까지 norm 구간 수: 9


In [20]:
print("\n클립 생성을 시작합니다...")

# 📁 결과 저장 폴더 구조 생성
os.makedirs('result/abnormal', exist_ok=True)
os.makedirs('result/normal', exist_ok=True)

# 이상행동 클립 저장
for i, item in enumerate(ab):
    try:
        video_base = os.path.splitext(item[0])[0]  # 확장자 제거한 영상 파일명
        ab_input_path = os.path.join(PATH_r, item[0])
        ab_output_path = f'result/abnormal/{i}_{video_base}_abnormal_10s.mp4'

        print(f"이상행동 클립 생성 중: {ab_output_path}")
        print(f"  입력: {ab_input_path}")
        print(f"  구간: {item[1]}초 ~ {item[2]}초")

        make_clip_video(ab_input_path, ab_output_path, item[1], item[2])

    except Exception as e:
        print(f"이상행동 클립 {i} 생성 중 오류 발생: {e}")
        print(f"파일 경로 확인: {ab_input_path}")
        print(f"파일 존재 여부: {os.path.exists(ab_input_path)}\n")

# 정상행동 클립 저장
for i, item in enumerate(norm):
    try:
        video_base = os.path.splitext(item[0])[0]  # 확장자 제거한 영상 파일명
        norm_input_path = os.path.join(PATH_r, item[0])
        norm_output_path = f'result/normal/{i}_{video_base}_normal_10s.mp4'

        print(f"정상행동 클립 생성 중: {norm_output_path}")
        print(f"  입력: {norm_input_path}")
        print(f"  구간: {item[1]}초 ~ {item[2]}초")

        make_clip_video(norm_input_path, norm_output_path, item[1], item[2])

    except Exception as e:
        print(f"정상행동 클립 {i} 생성 중 오류 발생: {e}")
        print(f"파일 경로 확인: {norm_input_path}")
        print(f"파일 존재 여부: {os.path.exists(norm_input_path)}\n")

print(f"\n모든 클립 생성 완료! result/abnormal 및 result/normal 폴더를 확인하세요.")



클립 생성을 시작합니다...
이상행동 클립 생성 중: result/abnormal/0_C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2_abnormal_10s.mp4
  입력: /home/kbj/dev_ws/unmanned_store/data/절도_mp4/C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2.mp4
  구간: 40초 ~ 50초
Moviepy - Building video result/abnormal/0_C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2_abnormal_10s.mp4.
Moviepy - Writing video result/abnormal/0_C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2_abnormal_10s.mp4



Moviepy - Done !
Moviepy - video ready result/abnormal/0_C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2_abnormal_10s.mp4
이상행동 클립 생성 중: result/abnormal/1_C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2_abnormal_10s.mp4
  입력: /home/kbj/dev_ws/unmanned_store/data/절도_mp4/C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2.mp4
  구간: 40초 ~ 50초
Moviepy - Building video result/abnormal/1_C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2_abnormal_10s.mp4.
Moviepy - Writing video result/abnormal/1_C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2_abnormal_10s.mp4



Moviepy - Done !
Moviepy - video ready result/abnormal/1_C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2_abnormal_10s.mp4
이상행동 클립 생성 중: result/abnormal/2_C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2_abnormal_10s.mp4
  입력: /home/kbj/dev_ws/unmanned_store/data/절도_mp4/C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2.mp4
  구간: 40초 ~ 50초
Moviepy - Building video result/abnormal/2_C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2_abnormal_10s.mp4.
Moviepy - Writing video result/abnormal/2_C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2_abnormal_10s.mp4



Moviepy - Done !
Moviepy - video ready result/abnormal/2_C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2_abnormal_10s.mp4
이상행동 클립 생성 중: result/abnormal/3_C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2_abnormal_10s.mp4
  입력: /home/kbj/dev_ws/unmanned_store/data/절도_mp4/C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2.mp4
  구간: 42초 ~ 52초
Moviepy - Building video result/abnormal/3_C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2_abnormal_10s.mp4.
Moviepy - Writing video result/abnormal/3_C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2_abnormal_10s.mp4



Moviepy - Done !
Moviepy - video ready result/abnormal/3_C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2_abnormal_10s.mp4
이상행동 클립 생성 중: result/abnormal/4_C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2_abnormal_10s.mp4
  입력: /home/kbj/dev_ws/unmanned_store/data/절도_mp4/C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2.mp4
  구간: 42초 ~ 52초
Moviepy - Building video result/abnormal/4_C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2_abnormal_10s.mp4.
Moviepy - Writing video result/abnormal/4_C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2_abnormal_10s.mp4



...

Moviepy - Done !
Moviepy - video ready result/normal/4_C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2_normal_10s.mp4
정상행동 클립 생성 중: result/normal/5_C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2_normal_10s.mp4
  입력: /home/kbj/dev_ws/unmanned_store/data/절도_mp4/C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2.mp4
  구간: 15초 ~ 25초
Moviepy - Building video result/normal/5_C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2_normal_10s.mp4.
Moviepy - Writing video result/normal/5_C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2_normal_10s.mp4



Moviepy - Done !
Moviepy - video ready result/normal/5_C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2_normal_10s.mp4
정상행동 클립 생성 중: result/normal/6_C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2_normal_10s.mp4
  입력: /home/kbj/dev_ws/unmanned_store/data/절도_mp4/C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2.mp4
  구간: 20초 ~ 30초
Moviepy - Building video result/normal/6_C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2_normal_10s.mp4.
Moviepy - Writing video result/normal/6_C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2_normal_10s.mp4



Moviepy - Done !
Moviepy - video ready result/normal/6_C_3_12_11_BU_DYA_07-27_13-03-09_CC_RGB_DF2_F2_normal_10s.mp4
정상행동 클립 생성 중: result/normal/7_C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2_normal_10s.mp4
  입력: /home/kbj/dev_ws/unmanned_store/data/절도_mp4/C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2.mp4
  구간: 20초 ~ 30초
Moviepy - Building video result/normal/7_C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2_normal_10s.mp4.
Moviepy - Writing video result/normal/7_C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2_normal_10s.mp4



Moviepy - Done !
Moviepy - video ready result/normal/7_C_3_12_11_BU_DYA_07-27_13-03-06_CA_RGB_DF2_F2_normal_10s.mp4
정상행동 클립 생성 중: result/normal/8_C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2_normal_10s.mp4
  입력: /home/kbj/dev_ws/unmanned_store/data/절도_mp4/C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2.mp4
  구간: 20초 ~ 30초
Moviepy - Building video result/normal/8_C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2_normal_10s.mp4.
Moviepy - Writing video result/normal/8_C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2_normal_10s.mp4



Moviepy - Done !
Moviepy - video ready result/normal/8_C_3_12_11_BU_DYA_07-27_13-03-09_CB_RGB_DF2_F2_normal_10s.mp4

모든 클립 생성 완료! result/abnormal 및 result/normal 폴더를 확인하세요.


In [10]:
# 생성된 파일 목록 확인
import os
result_files = os.listdir('result')
result_files.sort()

print(f"\n생성된 파일 목록 (총 {len(result_files)}개):")
for file in result_files:
    print(f"  - {file}")

# 이상행동/정상행동 구간 정보 출력
print(f"\n=== 이상행동 구간 정보 ===")
for i, info in enumerate(ab):
    print(f"{i}: 파일={info[0]}, 시작={info[1]}초, 끝={info[2]}초")

print(f"\n=== 정상행동 구간 정보 ===")
for i, info in enumerate(norm):
    print(f"{i}: 파일={info[0]}, 시작={info[1]}초, 끝={info[2]}초")


생성된 파일 목록 (총 2개):
  - abnormal
  - normal

=== 이상행동 구간 정보 ===
0: 파일=C_3_12_1_BU_SMB_08-28_16-25-26_CB_RGB_DF2_M1.mp4, 시작=24초, 끝=34초
1: 파일=C_3_12_1_BU_SMB_08-28_16-25-27_CC_RGB_DF2_M1.mp4, 시작=24초, 끝=34초
2: 파일=C_3_12_1_BU_SMB_08-28_16-25-26_CD_RGB_DF2_M1.mp4, 시작=24초, 끝=34초
3: 파일=C_3_12_1_BU_SMB_08-28_16-25-26_CA_RGB_DF2_M1.mp4, 시작=24초, 끝=34초
4: 파일=C_3_12_1_BU_SMB_08-28_16-25-26_CB_RGB_DF2_M1.mp4, 시작=32초, 끝=42초
5: 파일=C_3_12_1_BU_SMB_08-28_16-25-27_CC_RGB_DF2_M1.mp4, 시작=32초, 끝=42초
6: 파일=C_3_12_1_BU_SMB_08-28_16-25-26_CD_RGB_DF2_M1.mp4, 시작=32초, 끝=42초
7: 파일=C_3_12_1_BU_SMB_08-28_16-25-26_CA_RGB_DF2_M1.mp4, 시작=32초, 끝=42초
8: 파일=C_3_12_1_BU_SMB_08-28_16-25-26_CB_RGB_DF2_M1.mp4, 시작=42초, 끝=52초
9: 파일=C_3_12_1_BU_SMB_08-28_16-25-27_CC_RGB_DF2_M1.mp4, 시작=42초, 끝=52초
10: 파일=C_3_12_1_BU_SMB_08-28_16-25-26_CD_RGB_DF2_M1.mp4, 시작=42초, 끝=52초
11: 파일=C_3_12_1_BU_SMB_08-28_16-25-26_CA_RGB_DF2_M1.mp4, 시작=42초, 끝=52초

=== 정상행동 구간 정보 ===
0: 파일=C_3_12_1_BU_SMB_08-28_16-25-26_CB_RGB_DF2_M1.mp4, 시작=4초, 끝=14초
1: 파일=